# Advanced Topics and Optimization (EXTRA)

**Chapter 7: Predicting Soccer Outcomes with Deep Learning**

## Learning Objectives

- Understand regularization techniques (L1, L2, Dropout)
- Learn about batch normalization
- Explore advanced optimizers (Adam, RMSprop, AdamW)
- Implement learning rate scheduling
- Understand overfitting and how to prevent it
- Learn hyperparameter tuning strategies
- Explore model interpretation techniques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

## The Problem: Overfitting

When you train a soccer team, you don't just prepare them to beat one rival - you prepare them to face many opponents.

A model that only memorizes one opponent's game patterns will collapse when facing a different team. This is called **overfitting**: the model performs brilliantly on training data but poorly on new matches.

### Signs of Overfitting
- Training loss keeps dropping
- Validation loss starts rising
- Large gap between training and validation accuracy

### Solutions
1. **Regularization** (L1, L2, Dropout)
2. **More data**
3. **Simpler model**
4. **Early stopping**
5. **Data augmentation**

## Generate Data

In [ ]:
# Generate simulated match data
np.random.seed(42)
n_matches = 1000

shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

win_probability = 1 / (1 + np.exp(-(0.3 * (df['xg_home'] - df['xg_away']) + 
                                     0.02 * (df['shots_home'] - df['shots_away']) +
                                     0.01 * (df['possession_home'] - 50))))
df['home_win'] = (np.random.random(n_matches) < win_probability).astype(int)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Prepare data
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y = df['home_win'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# Further split training into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)}")
print(f"Validation set: {len(X_val)}")
print(f"Test set: {len(X_test)}")

---

# 1. Regularization Techniques

## L2 Regularization (Weight Decay)

**L2 regularization** (also called Ridge) softens all weights a little, making sure no single feature dominates.

Mathematically, it adds a penalty term to the loss:
$$L_{total} = L_{original} + \lambda \sum w_i^2$$

Where λ controls the strength of regularization.

In [ ]:
class MLPWithL2(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Create model
input_size = X_train.shape[1]
model_l2 = MLPWithL2(input_size)

# L2 regularization is implemented via weight_decay in optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer_l2 = optim.Adam(model_l2.parameters(), lr=0.001, weight_decay=0.01)  # L2 penalty

print(f"Model with L2 regularization (weight_decay=0.01)")
print(model_l2)

## Dropout Regularization

**Dropout** randomly "benches" some neurons during training, forcing the network to find multiple paths to a solution.

It's like a coach who rotates players so the team doesn't depend too much on one star striker.

In [ ]:
class MLPWithDropout(nn.Module):
    def __init__(self, input_size, hidden_size=64, dropout_rate=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)  # Drop 30% of neurons
        
        self.fc2 = nn.Linear(hidden_size, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)  # Randomly drop neurons
        
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        return x

model_dropout = MLPWithDropout(input_size, dropout_rate=0.3)
optimizer_dropout = optim.Adam(model_dropout.parameters(), lr=0.001)

print(f"Model with Dropout (rate=0.3)")
print(model_dropout)

## Batch Normalization

**Batch Normalization** normalizes the inputs to each layer, making training more stable and allowing higher learning rates.

It's like ensuring all players are warmed up and ready before each drill.

In [ ]:
class MLPWithBatchNorm(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)  # Batch normalization
        self.relu1 = nn.ReLU()
        
        self.fc2 = nn.Linear(hidden_size, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        
        self.fc3 = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)  # Normalize
        x = self.relu1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        
        x = self.fc3(x)
        return x

model_bn = MLPWithBatchNorm(input_size)
optimizer_bn = optim.Adam(model_bn.parameters(), lr=0.001)

print(f"Model with Batch Normalization")
print(model_bn)

## Combined: Dropout + Batch Norm + L2

Let's combine all regularization techniques for maximum robustness!

In [ ]:
class MLPRegularized(nn.Module):
    def __init__(self, input_size, hidden_size=64, dropout_rate=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(hidden_size, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        return x

model_reg = MLPRegularized(input_size, dropout_rate=0.3)
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=0.01)

print(f"Fully Regularized Model")
print(model_reg)

## Training Function with Validation Tracking

In [ ]:
def train_with_validation(model, optimizer, criterion, X_train, y_train, X_val, y_val, epochs=100):
    """
    Train model and track both training and validation metrics.
    """
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        
        logits = model(X_train).squeeze()
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()
        
        # Training metrics
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            train_acc = (preds == y_train).float().mean().item()
        
        train_losses.append(loss.item())
        train_accs.append(train_acc)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val).squeeze()
            val_loss = criterion(val_logits, y_val)
            val_probs = torch.sigmoid(val_logits)
            val_preds = (val_probs >= 0.5).float()
            val_acc = (val_preds == y_val).float().mean().item()
        
        val_losses.append(val_loss.item())
        val_accs.append(val_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} - "
                  f"Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}, "
                  f"Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")
    
    return train_losses, val_losses, train_accs, val_accs

In [ ]:
# Train regularized model
print("Training Fully Regularized Model...")
train_losses, val_losses, train_accs, val_accs = train_with_validation(
    model_reg, optimizer_reg, criterion, X_train, y_train, X_val, y_val, epochs=100
)

In [ ]:
# Visualize training vs validation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(train_accs, label='Training Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: Small gap between train and val = good generalization!")

---

# 2. Learning Rate Scheduling

The **learning rate** is one of the most important hyperparameters. Too high and training is unstable; too low and it's painfully slow.

**Learning rate scheduling** adjusts the learning rate during training:
- Start high for fast initial progress
- Reduce gradually to fine-tune

Like a coach who pushes hard in pre-season but eases off before big matches.

In [ ]:
# Model with learning rate scheduling
model_sched = MLPRegularized(input_size)
optimizer_sched = optim.Adam(model_sched.parameters(), lr=0.01)  # Start with higher LR

# Learning rate scheduler - reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_sched, 
    mode='min',      # minimize validation loss
    factor=0.5,      # multiply LR by 0.5
    patience=10,     # wait 10 epochs before reducing
    verbose=True
)

print("Model with Learning Rate Scheduling")
print(f"Initial learning rate: {optimizer_sched.param_groups[0]['lr']}")

In [ ]:
# Training with scheduler
val_losses_sched = []
lr_history = []

for epoch in range(100):
    model_sched.train()
    optimizer_sched.zero_grad()
    
    logits = model_sched(X_train).squeeze()
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer_sched.step()
    
    # Validation
    model_sched.eval()
    with torch.no_grad():
        val_logits = model_sched(X_val).squeeze()
        val_loss = criterion(val_logits, y_val)
    
    val_losses_sched.append(val_loss.item())
    lr_history.append(optimizer_sched.param_groups[0]['lr'])
    
    # Update learning rate based on validation loss
    scheduler.step(val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} - Val Loss: {val_loss.item():.4f}, LR: {optimizer_sched.param_groups[0]['lr']:.6f}")

In [ ]:
# Visualize learning rate changes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(val_losses_sched, linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation Loss', fontsize=12)
axes[0].set_title('Validation Loss with LR Scheduling', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_history, linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# 3. Advanced Optimizers

We've been using **Adam** (Adaptive Moment Estimation), but there are other options:

1. **SGD (Stochastic Gradient Descent)**: Classic, simple, requires careful tuning
2. **Adam**: Adaptive learning rates, works well out-of-the-box
3. **RMSprop**: Good for RNNs
4. **AdamW**: Adam with better weight decay
5. **AdaGrad**: Adapts learning rate per parameter

Let's compare a few!

In [ ]:
# Compare different optimizers
optimizers_to_test = {
    'SGD': optim.SGD,
    'Adam': optim.Adam,
    'AdamW': optim.AdamW,
    'RMSprop': optim.RMSprop
}

results = {}

for opt_name, opt_class in optimizers_to_test.items():
    print(f"\nTraining with {opt_name}...")
    
    # Create fresh model
    model = MLPRegularized(input_size)
    
    # Create optimizer
    if opt_name == 'SGD':
        optimizer = opt_class(model.parameters(), lr=0.01, momentum=0.9)
    else:
        optimizer = opt_class(model.parameters(), lr=0.001)
    
    # Train
    val_losses_opt = []
    for epoch in range(50):  # Shorter training for comparison
        model.train()
        optimizer.zero_grad()
        logits = model(X_train).squeeze()
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val).squeeze()
            val_loss = criterion(val_logits, y_val)
        
        val_losses_opt.append(val_loss.item())
    
    results[opt_name] = val_losses_opt
    print(f"  Final validation loss: {val_losses_opt[-1]:.4f}")

In [ ]:
# Visualize optimizer comparison
plt.figure(figsize=(10, 6))
for opt_name, losses in results.items():
    plt.plot(losses, label=opt_name, linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.title('Optimizer Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\nNote: Adam and AdamW typically converge faster than SGD")

---

# 4. Hyperparameter Tuning

Finding the best hyperparameters is crucial for model performance.

### Key Hyperparameters
1. **Learning rate**: Most important!
2. **Batch size**: 16, 32, 64, 128
3. **Hidden layer size**: 16, 32, 64, 128
4. **Number of layers**: 1, 2, 3, 4
5. **Dropout rate**: 0.1, 0.2, 0.3, 0.5
6. **Weight decay**: 0.0001, 0.001, 0.01

### Tuning Strategies
1. **Grid Search**: Try all combinations (slow but thorough)
2. **Random Search**: Random sampling (faster, often as good)
3. **Bayesian Optimization**: Smart search (best for expensive models)
4. **Manual Tuning**: Start here! Understand each parameter

In [ ]:
# Simple grid search example
hyperparams = {
    'hidden_size': [16, 32, 64],
    'dropout_rate': [0.1, 0.3, 0.5],
    'learning_rate': [0.0001, 0.001, 0.01]
}

best_val_loss = float('inf')
best_params = {}

print("Running hyperparameter search (simplified)...\n")

# Sample a few random combinations
import itertools
all_combinations = list(itertools.product(
    hyperparams['hidden_size'],
    hyperparams['dropout_rate'],
    hyperparams['learning_rate']
))

# Test first 5 combinations (in practice, test all or use random search)
for hidden_size, dropout_rate, lr in all_combinations[:5]:
    print(f"Testing: hidden={hidden_size}, dropout={dropout_rate}, lr={lr}")
    
    model = MLPRegularized(input_size, hidden_size=hidden_size, dropout_rate=dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Quick training (20 epochs)
    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train).squeeze()
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val).squeeze()
        val_loss = criterion(val_logits, y_val).item()
    
    print(f"  Validation loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = {'hidden_size': hidden_size, 'dropout_rate': dropout_rate, 'lr': lr}
    
    print()

print(f"\nBest parameters found:")
for param, value in best_params.items():
    print(f"  {param}: {value}")
print(f"Best validation loss: {best_val_loss:.4f}")

---

# 5. Model Interpretation

## Feature Importance

Which features are most important for predictions? Let's analyze!

In [ ]:
# Train a simple model for interpretation
model_interp = MLPRegularized(input_size)
optimizer_interp = optim.Adam(model_interp.parameters(), lr=0.001, weight_decay=0.01)

for epoch in range(100):
    model_interp.train()
    optimizer_interp.zero_grad()
    logits = model_interp(X_train).squeeze()
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer_interp.step()

print("Model trained for interpretation")

In [ ]:
# Analyze first layer weights
first_layer_weights = model_interp.fc1.weight.data.abs().mean(dim=0).numpy()

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_cols, first_layer_weights)
plt.xlabel('Average Absolute Weight', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance (First Layer Weights)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 3 most important features:")
top_indices = np.argsort(first_layer_weights)[-3:][::-1]
for idx in top_indices:
    print(f"  {feature_cols[idx]}: {first_layer_weights[idx]:.4f}")

## Gradient-Based Feature Importance

In [ ]:
# Calculate gradients with respect to inputs
model_interp.eval()
X_sample = X_test[:100].clone().requires_grad_(True)

logits = model_interp(X_sample).squeeze()
logits.sum().backward()

# Gradient magnitude = feature importance
feature_importance = X_sample.grad.abs().mean(dim=0).numpy()

plt.figure(figsize=(10, 6))
plt.barh(feature_cols, feature_importance, color='orange')
plt.xlabel('Average Gradient Magnitude', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Gradient-Based Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

✅ **Regularization techniques** to prevent overfitting:
- L2 regularization (weight decay)
- Dropout
- Batch normalization

✅ **Learning rate scheduling** for better convergence

✅ **Different optimizers** and when to use them

✅ **Hyperparameter tuning** strategies

✅ **Model interpretation** techniques:
- Weight analysis
- Gradient-based importance

### Best Practices

1. **Always use validation set** to detect overfitting
2. **Start simple** - add complexity only if needed
3. **Regularize** - dropout + weight decay work well together
4. **Tune learning rate first** - most important hyperparameter
5. **Use early stopping** - don't overtrain
6. **Monitor metrics** - plot training curves
7. **Interpret results** - understand what the model learned

### Recommended Workflow

1. Start with simple model
2. Add regularization if overfitting
3. Tune learning rate
4. Experiment with architecture
5. Try different optimizers
6. Fine-tune other hyperparameters
7. Validate on test set only once at the end

## Practice Exercises

1. **Experiment with regularization**: Try different dropout rates (0.1, 0.3, 0.5, 0.7). What happens with very high dropout?

2. **Learning rate finder**: Implement a learning rate finder that tries many learning rates and plots the loss.

3. **Cosine annealing**: Implement cosine annealing learning rate schedule.

4. **Layer-wise learning rates**: Use different learning rates for different layers.

5. **Ensemble methods**: Train multiple models and average their predictions.

6. **Cross-validation**: Implement k-fold cross-validation for more robust evaluation.

7. **Advanced interpretation**: Try SHAP values or integrated gradients for feature importance.

8. **Automated tuning**: Use a library like Optuna or Ray Tune for hyperparameter optimization.